In [1]:
#Libraries
import os
import zipfile
import urllib.request

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
#Download & set Dataset
dataset_url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
zip_path = "ml-latest-small.zip"
extract_folder = "ml-latest-small"

if not os.path.exists(zip_path):
    urllib.request.urlretrieve(dataset_url, zip_path)
    print("Dataset downloaded successfully")
else:
    print("Dataset already downloaded")

if not os.path.exists(extract_folder):
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall()
    print("Dataset extracted successfully")
else:
    print("Dataset already extracted")

Dataset downloaded successfully
Dataset extracted successfully


In [3]:
#Load data
movies = pd.read_csv("ml-latest-small/movies.csv")
ratings = pd.read_csv("ml-latest-small/ratings.csv")

print("Movies data shape:", movies.shape)
print("Ratings data shape:", ratings.shape)

display(movies.head())
display(ratings.head())

Movies data shape: (9742, 3)
Ratings data shape: (100836, 4)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [4]:
print("Total movies:", movies["movieId"].nunique())
print("Total users:", ratings["userId"].nunique())
print("Total ratings:", len(ratings))

print("\nRating values:")
print(sorted(ratings["rating"].unique()))

Total movies: 9742
Total users: 610
Total ratings: 100836

Rating values:
[np.float64(0.5), np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0), np.float64(3.5), np.float64(4.0), np.float64(4.5), np.float64(5.0)]


In [5]:
rating_stats = ratings.groupby("movieId").agg(
    average_rating=("rating", "mean"),
    rating_count=("rating", "count")
).reset_index()

movies = movies.merge(rating_stats, on="movieId", how="left")

movies["average_rating"] = movies["average_rating"].fillna(0)
movies["rating_count"] = movies["rating_count"].fillna(0)

display(movies.head())

,movieId,title,genres,average_rating,rating_count
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.920930,215.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.431818,110.0
2,3,Grumpier Old Men (1995),Comedy|Romance,3.259615,52.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,2.357143,7.0
4,5,Father of the Bride Part II (1995),Comedy,3.071429,49.0


In [6]:
movies["clean_genres"] = movies["genres"].str.replace("|", " ", regex=False)
movies["clean_genres"] = movies["clean_genres"].replace("(no genres listed)", "")

display(movies[["movieId", "title", "genres", "clean_genres"]].head())

,movieId,title,genres,clean_genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,Comedy


In [7]:
#Create cosine similarity matrix
vectorizer = CountVectorizer()
genre_matrix = vectorizer.fit_transform(movies["clean_genres"])

cosine_sim = cosine_similarity(genre_matrix)

print("Cosine similarity matrix created")
print("Matrix shape:", cosine_sim.shape)

Cosine similarity matrix created
Matrix shape: (9742, 9742)


In [8]:
#Search movies by name
def search_movies(keyword):
    results = movies[movies["title"].str.contains(keyword, case=False, na=False)]
    return results[["movieId", "title", "genres", "average_rating", "rating_count"]].head(10)

search_movies("toy story")

,movieId,title,genres,average_rating,rating_count
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.920930,215.0
2355,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,3.860825,97.0
7355,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,4.109091,55.0


In [9]:
#Content Based Recommender using cosine Similarity
def recommend_by_cosine(movie_title, top_n=5):
    matched_movies = movies[movies["title"].str.lower() == movie_title.lower()]

    if matched_movies.empty:
        print("Movie not found. Use search_movies() to check the exact title.")
        return None

    movie_index = matched_movies.index[0]
    selected_movie = movies.loc[movie_index]

    similarity_scores = list(enumerate(cosine_sim[movie_index]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    recommendations = []

    for index, score in similarity_scores:
        if index == movie_index:
            continue

        movie = movies.loc[index]

        if movie["rating_count"] < 20:
            continue

        selected_genres = set(selected_movie["clean_genres"].split())
        recommended_genres = set(movie["clean_genres"].split())
        common_genres = selected_genres.intersection(recommended_genres)

        explanation = "Similar genres: " + ", ".join(common_genres)

        recommendations.append({
            "title": movie["title"],
            "genres": movie["genres"],
            "similarity_score": round(score, 3),
            "average_rating": round(movie["average_rating"], 2),
            "rating_count": int(movie["rating_count"]),
            "explanation": explanation
        })

        if len(recommendations) == top_n:
            break

    return pd.DataFrame(recommendations)

In [10]:
#Test
selected_movie = "Toy Story (1995)"

cosine_recommendations = recommend_by_cosine(selected_movie, top_n=5)

print("Selected Movie:", selected_movie)
display(cosine_recommendations)

Selected Movie: Toy Story (1995)


,title,genres,similarity_score,average_rating,rating_count,explanation
0,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.0,3.24,45,"Similar genres: Comedy, Fantasy, Adventure, Ch..."
1,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.0,3.86,97,"Similar genres: Comedy, Fantasy, Adventure, Ch..."
2,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0,3.72,37,"Similar genres: Comedy, Fantasy, Adventure, Ch..."
3,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,1.0,3.87,132,"Similar genres: Comedy, Fantasy, Adventure, Ch..."
4,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,1.0,3.02,21,"Similar genres: Comedy, Fantasy, Adventure, Ch..."


In [11]:
#Data Preperation for Pearson correlation
popular_movie_ids = movies[movies["rating_count"] >= 50]["movieId"]

popular_ratings = ratings[ratings["movieId"].isin(popular_movie_ids)]

movie_user_matrix = popular_ratings.pivot_table(
    index="movieId",
    columns="userId",
    values="rating"
)

pearson_sim = movie_user_matrix.T.corr(min_periods=10)

print("Pearson correlation matrix created")
print("Matrix shape:", pearson_sim.shape)

Pearson correlation matrix created
Matrix shape: (450, 450)


In [12]:
#Recommender using Pearson correlation
def recommend_by_pearson(movie_title, top_n=5):
    matched_movies = movies[movies["title"].str.lower() == movie_title.lower()]

    if matched_movies.empty:
        print("Movie not found. Use search_movies() to check the exact title.")
        return None

    selected_movie_id = matched_movies.iloc[0]["movieId"]

    if selected_movie_id not in pearson_sim.index:
        print("This movie does not have enough ratings for Pearson correlation.")
        return None

    similar_movies = pearson_sim[selected_movie_id].dropna()
    similar_movies = similar_movies.drop(selected_movie_id)
    similar_movies = similar_movies.sort_values(ascending=False)

    recommendations = []

    for movie_id, score in similar_movies.items():
        movie = movies[movies["movieId"] == movie_id].iloc[0]

        if score <= 0:
            continue

        explanation = "Users rated this movie in a similar pattern."

        recommendations.append({
            "title": movie["title"],
            "genres": movie["genres"],
            "pearson_score": round(score, 3),
            "average_rating": round(movie["average_rating"], 2),
            "rating_count": int(movie["rating_count"]),
            "explanation": explanation
        })

        if len(recommendations) == top_n:
            break

    return pd.DataFrame(recommendations)

In [13]:
#Test
selected_movie = "Toy Story (1995)"

pearson_recommendations = recommend_by_pearson(selected_movie, top_n=5)

print("Selected Movie:", selected_movie)
display(pearson_recommendations)

Selected Movie: Toy Story (1995)


,title,genres,pearson_score,average_rating,rating_count,explanation
0,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.699,3.86,97,Users rated this movie in a similar pattern.
1,Arachnophobia (1990),Comedy|Horror,0.652,2.81,53,Users rated this movie in a similar pattern.
2,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy,0.643,3.84,125,Users rated this movie in a similar pattern.
3,Finding Nemo (2003),Adventure|Animation|Children|Comedy,0.619,3.96,141,Users rated this movie in a similar pattern.
4,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical,0.612,3.79,183,Users rated this movie in a similar pattern.


In [14]:
selected_movie = "Toy Story (1995)"

print("Content-Based Recommendations using Cosine Similarity")
display(recommend_by_cosine(selected_movie, top_n=5))

print("\nCollaborative Recommendations using Pearson Correlation")
display(recommend_by_pearson(selected_movie, top_n=5))

Content-Based Recommendations using Cosine Similarity


,title,genres,similarity_score,average_rating,rating_count,explanation
0,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.0,3.24,45,"Similar genres: Comedy, Fantasy, Adventure, Ch..."
1,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.0,3.86,97,"Similar genres: Comedy, Fantasy, Adventure, Ch..."
2,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0,3.72,37,"Similar genres: Comedy, Fantasy, Adventure, Ch..."
3,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,1.0,3.87,132,"Similar genres: Comedy, Fantasy, Adventure, Ch..."
4,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,1.0,3.02,21,"Similar genres: Comedy, Fantasy, Adventure, Ch..."



Collaborative Recommendations using Pearson Correlation


,title,genres,pearson_score,average_rating,rating_count,explanation
0,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.699,3.86,97,Users rated this movie in a similar pattern.
1,Arachnophobia (1990),Comedy|Horror,0.652,2.81,53,Users rated this movie in a similar pattern.
2,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy,0.643,3.84,125,Users rated this movie in a similar pattern.
3,Finding Nemo (2003),Adventure|Animation|Children|Comedy,0.619,3.96,141,Users rated this movie in a similar pattern.
4,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical,0.612,3.79,183,Users rated this movie in a similar pattern.


In [15]:
search_movies("matrix")

,movieId,title,genres,average_rating,rating_count
1939,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.192446,278.0
4351,6365,"Matrix Reloaded, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX,3.354167,96.0
4639,6934,"Matrix Revolutions, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX,3.151899,79.0
5669,27660,"Animatrix, The (2003)",Action|Animation|Drama|Sci-Fi,3.700000,20.0


In [16]:
selected_movie = "Matrix, The (1999)"

print("Content-Based Recommendations using Cosine Similarity")
display(recommend_by_cosine(selected_movie, top_n=5))

print("\nCollaborative Recommendations using Pearson Correlation")
display(recommend_by_pearson(selected_movie, top_n=5))

Content-Based Recommendations using Cosine Similarity


,title,genres,similarity_score,average_rating,rating_count,explanation
0,Johnny Mnemonic (1995),Action|Sci-Fi|Thriller,1.0,2.68,53,"Similar genres: Thriller, Sci-Fi, Action"
1,Timecop (1994),Action|Sci-Fi|Thriller,1.0,2.50,21,"Similar genres: Thriller, Sci-Fi, Action"
2,Blade Runner (1982),Action|Sci-Fi|Thriller,1.0,4.10,124,"Similar genres: Thriller, Sci-Fi, Action"
3,"Arrival, The (1996)",Action|Sci-Fi|Thriller,1.0,3.11,22,"Similar genres: Thriller, Sci-Fi, Action"
4,"Terminator, The (1984)",Action|Sci-Fi|Thriller,1.0,3.90,131,"Similar genres: Thriller, Sci-Fi, Action"



Collaborative Recommendations using Pearson Correlation


,title,genres,pearson_score,average_rating,rating_count,explanation
0,Tommy Boy (1995),Comedy,0.675,3.78,50,Users rated this movie in a similar pattern.
1,Slumdog Millionaire (2008),Crime|Drama|Romance,0.614,3.81,71,Users rated this movie in a similar pattern.
2,Kung Fu Panda (2008),Action|Animation|Children|Comedy|IMAX,0.613,3.44,54,Users rated this movie in a similar pattern.
3,Interstellar (2014),Sci-Fi|IMAX,0.599,3.99,73,Users rated this movie in a similar pattern.
4,Legends of the Fall (1994),Drama|Romance|War|Western,0.567,3.40,68,Users rated this movie in a similar pattern.


In [17]:
selected_movie = "Toy Story (1995)"

final_recommendations = recommend_by_cosine(selected_movie, top_n=10)
final_recommendations.to_csv("movie_recommendations.csv", index=False)

print("Recommendations saved as movie_recommendations.csv")

Recommendations saved as movie_recommendations.csv
